# **Bonus Project:** Classifying NSL-KDD Dataset using DRL

Classify network traffic from the NSL-KDD dataset as normal (0) or anomalous (1) using Deep Reinforcement Learning (DRL).

A Deep Q-Network (DQN) will be used to train a reinforcement learning agent, which learns to classify traffic based on feedback from its actions.

The model's goal is to optimize its policy and accurately identify normal vs. attack traffic for intrusion detection.


## Exploring Phase

In [4]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv('../data/KDD_Shuffled_Combined_Set.csv')

In [6]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,private,REJ,0,0,0,0,0,0,...,10,0.04,0.06,0.00,0.00,0.00,0.0,1.00,1.00,1.0
1,0,tcp,private,REJ,0,0,0,0,0,0,...,1,0.00,0.06,0.00,0.00,0.00,0.0,1.00,1.00,1.0
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,86,0.61,0.04,0.61,0.02,0.00,0.0,0.00,0.00,0.0
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,57,1.00,0.00,1.00,0.28,0.00,0.0,0.00,0.00,1.0
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,86,0.31,0.17,0.03,0.02,0.00,0.0,0.83,0.71,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22538,0,tcp,smtp,SF,794,333,0,0,0,0,...,141,0.72,0.06,0.01,0.01,0.01,0.0,0.00,0.00,0.0
22539,0,tcp,http,SF,317,938,0,0,0,0,...,255,1.00,0.00,0.01,0.01,0.01,0.0,0.00,0.00,0.0
22540,0,tcp,http,SF,54540,8314,0,0,0,2,...,255,1.00,0.00,0.00,0.00,0.00,0.0,0.07,0.07,1.0
22541,0,udp,domain_u,SF,42,42,0,0,0,0,...,252,0.99,0.01,0.00,0.00,0.00,0.0,0.00,0.00,0.0


In [7]:
df.isnull().sum()

,0
duration,0
protocol_type,0
service,0
flag,0
src_bytes,0
dst_bytes,0
land,0
wrong_fragment,0
urgent,0
hot,0


In [8]:
df.dtypes

,0
duration,int64
protocol_type,object
service,object
flag,object
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
urgent,int64
hot,int64


In [9]:
for col_name in ('label', 'protocol_type', 'service', 'flag'):
    unique = df[col_name].unique()
    print(col_name, '\t', len(unique), unique)
    print()

label 	 2 [1. 0.]

protocol_type 	 3 ['tcp' 'icmp' 'udp']

service 	 63 ['private' 'ftp_data' 'eco_i' 'telnet' 'http' 'smtp' 'ftp' 'ldap' 'pop_3'
 'courier' 'discard' 'ecr_i' 'imap4' 'domain_u' 'mtp' 'systat' 'iso_tsap'
 'other' 'csnet_ns' 'finger' 'uucp' 'whois' 'netbios_ns' 'link' 'Z39_50'
 'sunrpc' 'auth' 'netbios_dgm' 'uucp_path' 'vmnet' 'domain' 'name' 'pop_2'
 'http_443' 'urp_i' 'login' 'gopher' 'exec' 'time' 'remote_job' 'ssh'
 'kshell' 'sql_net' 'shell' 'hostnames' 'echo' 'daytime' 'pm_dump' 'IRC'
 'netstat' 'ctf' 'nntp' 'netbios_ssn' 'tim_i' 'supdup' 'bgp' 'nnsp' 'rje'
 'printer' 'efs' 'X11' 'ntp_u' 'klogin']

flag 	 11 ['REJ' 'SF' 'RSTO' 'S0' 'RSTR' 'SH' 'S3' 'S2' 'S1' 'RSTOS0' 'OTH']



## Preprocessing Phase

In [10]:

col_names = ['protocol_type', 'service', 'flag']

df_encoded = pd.get_dummies(df, columns=col_names)
df_encoded

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
1,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
2,2,12983,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
3,0,20,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
4,1,0,15,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22538,0,794,333,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
22539,0,317,938,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
22540,0,54540,8314,0,0,0,2,0,1,1,...,False,False,False,False,False,False,False,False,True,False
22541,0,42,42,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [11]:
df_encoded.dtypes

,0
duration,int64
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
...,...
flag_S1,bool
flag_S2,bool
flag_S3,bool
flag_SF,bool


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)
df_scaled['label'] = df_encoded['label']
df_scaled['label']

,label
0,1.0
1,1.0
2,0.0
3,1.0
4,1.0
...,...
22538,0.0
22539,0.0
22540,1.0
22541,0.0


In [13]:
from sklearn.model_selection import train_test_split

X = df_scaled.drop('label', axis=1)
y = df_scaled['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Creating Environment

In [32]:
import gym
from gym import spaces
import random

class NSLKDDEnv(gym.Env):
    def __init__(self, X, y):
        super(NSLKDDEnv, self).__init__()

        # Attached dataset can be test or train depending on the phase
        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)

        # Initial State
        self.reset()

        # Actions are Normal and Anomaly
        self.action_space = spaces.Discrete(2)

        # Space is simply the features
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.X.shape[1],), dtype=np.float32)

    def reset(self):
        # Randomly select any row from the linked dataset
        self.loc = 0
        self.state = self.X.loc[self.loc]
        return self.state

    def step(self, action):
        # Postive reward if correct action
        # Negative reward if wrong action
        reward = 1 if action == self.y.loc[self.loc] else -1

        # Next location
        self.loc += 1

        # Done if there is no more remaining locations
        done = True
        if self.loc < self.X.shape[0]:
            self.state = self.X.loc[self.loc]
            done = False

        return self.state, reward, done, {}

    def render(self):
        print(f'NSLKDDEnv: location {self.loc}')

In [34]:
env = NSLKDDEnv(X_train, y_train)

state = env.reset()
done = False
score = 0
while not done:
    action = env.action_space.sample()  # Random action (select feature index)
    new_state, reward, done, _ = env.step(action)
    score += reward
    print(f'{env.loc:-4}- score {score:-4}')
    state = new_state
    if env.loc == 10:
        break

   1- score   -1
   2- score   -2
   3- score   -1
   4- score   -2
   5- score   -3
   6- score   -2
   7- score   -1
   8- score    0
   9- score    1
  10- score    0


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Training Phase, DRL Model

In [35]:
import tensorflow as tf
from tensorflow.keras import models, layers, optimizers

In [36]:
input_shape = env.observation_space.shape[0]
num_actions = env.action_space.n
input_shape, num_actions

(115, 2)

In [37]:
q_model = models.Sequential([
    layers.Input(shape=(input_shape,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_actions, activation='linear')
])

In [38]:
q_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │           7,424 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,714 (45.76 KB)

 Trainable params: 11,714 (45.76 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
!pip install keras
!pip install keras-rl

In [40]:
!pip install stable-baselines3 torch shimmy

In [41]:
from stable_baselines3 import PPO

In [49]:
env = NSLKDDEnv(X_train, y_train)
model = PPO('MlpPolicy', env, verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [50]:
model.learn(total_timesteps=100000)

-----------------------------
| time/              |      |
|    fps             | 885  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 519        |
|    iterations           | 2          |
|    time_elapsed         | 7          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01755197 |
|    clip_fraction        | 0.343      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.68      |
|    explained_variance   | -0.0451    |
|    learning_rate        | 0.0003     |
|    loss                 | 4.95       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0453    |
|    value_loss           | 8.8        |
----------------------------------------


KeyboardInterrupt: 

In [45]:
model.save("drl_model")

In [54]:
model = PPO.load("drl_model")

# Evaluate the agent
envs = {'train': NSLKDDEnv(X_train, y_train), 'test': NSLKDDEnv(X_test, y_test)}
for phase in ['train', 'test']:
    print(f"Evaluating {phase} phase...")

    env = envs[phase]
    obs = env.reset()
    done = False
    total_rewards = 0
    step = 0

    while not done:
        action, _state = model.predict(obs)
        obs, reward, done, info = env.step(action)
        total_rewards += reward
        step += 1
        if step % 1000 == 0:
            print(f"Step: {step:-10}, Total Rewards: {total_rewards}")

    print(f"Total Rewards of {phase}: {total_rewards}")
    print(f"Accuracy of {phase}: {total_rewards / env.X.shape[0]}")

    env.close()

Evaluating train phase...
Step:       1000, Total Rewards: 918
Step:       2000, Total Rewards: 1854
Step:       3000, Total Rewards: 2770
Step:       4000, Total Rewards: 3682
Step:       5000, Total Rewards: 4622
Step:       6000, Total Rewards: 5556
Step:       7000, Total Rewards: 6492
Step:       8000, Total Rewards: 7398
Step:       9000, Total Rewards: 8342
Step:      10000, Total Rewards: 9274
Step:      11000, Total Rewards: 10204
Step:      12000, Total Rewards: 11132
Step:      13000, Total Rewards: 12056
Step:      14000, Total Rewards: 12992
Step:      15000, Total Rewards: 13912
Step:      16000, Total Rewards: 14862
Step:      17000, Total Rewards: 15796
Step:      18000, Total Rewards: 16718
Total Rewards of train: 16750
Accuracy of train: 0.9288011533769547
Evaluating test phase...
Step:       1000, Total Rewards: 908
Step:       2000, Total Rewards: 1834
Step:       3000, Total Rewards: 2770
Step:       4000, Total Rewards: 3674
Total Rewards of test: 4143
Accuracy of